# Code 3: Prompt Repetition Experiment

Tests the effect of repeating the prompt (query duplication) on NLI.

Experiments:
  A) Pure LLM: Zero-Shot Repetition
  B) Pure LLM: Few-Shot Repetition
  C) Hybrid: CE + Zero-Shot Repetition fallback
  D) Hybrid: CE + Few-Shot Repetition fallback


In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder
from google import genai
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ============================================================
# Configuration
# ============================================================
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["GOOGLE_API_KEY"] = "Your API Key Here"  # Set your API key in environment variable

CE_MODEL_NAME = "cross-encoder/nli-distilroberta-base"
LLM_MODEL = "gemini-2.0-flash"
DATA_FILE = "multinli_1.0_dev_matched.jsonl"
TEST_SIZE = 2000          # Change to 2000 for final run
FALLBACK_THRESHOLD = 0.85
MAX_RETRIES = 6
REQUEST_DELAY = 4
VALID_LABELS = ["contradiction", "entailment", "neutral"]

COST_INPUT = 0.1   # USD per 1M tokens
COST_OUTPUT = 0.4

# ============================================================
# Data Loading
# ============================================================
def load_data(path, n):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(samples) >= n:
                break
            try:
                obj = json.loads(line.strip())
                if obj.get("gold_label") in VALID_LABELS:
                    samples.append(obj)
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(samples)} samples.")
    return samples

# ============================================================
# Repetition Prompt Builders
# The core idea: duplicate the entire prompt separated by newlines.
# No examples added, no reasoning requested — pure input repetition.
# ============================================================
def build_zero_shot_base(premise, hypothesis):
    return (
        "You are an NLI classifier. Determine if the hypothesis is an "
        "'entailment', 'contradiction', or 'neutral' to the premise.\n"
        "- entailment: the hypothesis MUST be true given the premise.\n"
        "- contradiction: the hypothesis CANNOT be true given the premise.\n"
        "- neutral: the hypothesis MIGHT be true, but the premise doesn't "
        "give enough info to be sure.\n\n"
        "Respond with EXACTLY ONE word. No punctuation. No explanation.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n\n"
        "Label:"
    )

def build_few_shot_base(premise, hypothesis):
    examples = (
        "Example 1:\n"
        "Premise: \"The program has been in place since 1994 and has helped "
        "thousands of low-income families.\"\n"
        "Hypothesis: \"A program has provided assistance to families in need.\"\n"
        "Label: entailment\n\n"
        "Example 2:\n"
        "Premise: \"He turned around and settled into his chair, looking out "
        "the window at the garden.\"\n"
        "Hypothesis: \"He was thinking about his childhood when he looked "
        "out the window.\"\n"
        "Label: neutral\n\n"
        "Example 3:\n"
        "Premise: \"The new rights are nice enough, but they do not like to "
        "appear to be asked.\"\n"
        "Hypothesis: \"Everyone has always appreciated being asked about "
        "new rights.\"\n"
        "Label: contradiction\n"
    )
    return (
        "You are an expert NLI classifier.\n"
        "Classify the relationship between premise and hypothesis as: "
        "entailment, contradiction, or neutral.\n\n"
        f"{examples}\n"
        "Classify the following pair. Respond with EXACTLY ONE word.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n"
        "Label:"
    )

def build_repetition(premise, hypothesis, base_builder):
    """Apply query repetition: duplicate the full prompt."""
    base = base_builder(premise, hypothesis)
    return base + "\n\n" + base

# ============================================================
# Cross-Encoder
# ============================================================
def run_cross_encoder(samples):
    print("Loading Cross-Encoder...")
    ce = CrossEncoder(CE_MODEL_NAME, device="cpu")
    pairs = [(s["sentence1"], s["sentence2"]) for s in samples]
    scores = ce.predict(pairs, show_progress_bar=True)

    preds, confs = [], []
    for score in scores:
        probs = np.exp(score - np.max(score))
        probs = probs / probs.sum()
        preds.append(VALID_LABELS[np.argmax(probs)])
        confs.append(np.max(probs))
    return np.array(preds), np.array(confs)

# ============================================================
# LLM Call + Parse
# ============================================================
def call_llm(client, prompt, max_tokens=10):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config={"temperature": 0, "max_output_tokens": max_tokens}
            )
            raw = response.text.strip() if response.text else ""
            in_tok = getattr(response.usage_metadata, "prompt_token_count", 0) or 0
            out_tok = getattr(response.usage_metadata, "candidates_token_count", 0) or 0
            return raw, in_tok, out_tok
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f"  API error (attempt {attempt}): {e}, retrying in {wait}s")
            time.sleep(wait)
    return "error", 0, 0

def parse_label(raw):
    if not raw or "error" in raw:
        return "error"
    matches = re.findall(r"\b(entailment|contradiction|neutral)\b", raw.lower())
    return matches[-1] if matches else "unknown"

# ============================================================
# Error Analysis
# ============================================================
def print_error_analysis(samples, gold, pred, name, top_n=10):
    print(f"\n--- Error Analysis: {name} ---")
    error_types = defaultdict(int)
    errors = []
    for i, (g, p) in enumerate(zip(gold, pred)):
        if g != p:
            error_types[f"{g} -> {p}"] += 1
            errors.append((i, g, p))

    print(f"Total errors: {len(errors)} / {len(gold)}")
    print("Error type distribution:")
    for t, c in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f"  {t}: {c}")

    print(f"\nTop {min(top_n, len(errors))} error samples:")
    for idx, g, p in errors[:top_n]:
        print(f"  [{idx}] {g} -> {p} | "
              f"P: {samples[idx]['sentence1'][:60]}... | "
              f"H: {samples[idx]['sentence2'][:60]}...")

# ============================================================
# Experiment Runner
# ============================================================
def run_pure_llm_repetition(client, samples, name, base_builder):
    """Experiment A: Pure LLM with query repetition on all samples."""
    y_test = [s["gold_label"] for s in samples]
    preds, total_in, total_out = [], 0, 0

    for i, s in enumerate(tqdm(samples, desc=name)):
        prompt = build_repetition(s["sentence1"], s["sentence2"], base_builder)
        raw, in_tok, out_tok = call_llm(client, prompt)
        preds.append(parse_label(raw))
        total_in += in_tok
        total_out += out_tok
        time.sleep(REQUEST_DELAY)

    acc = accuracy_score(y_test, preds)
    cost = (total_in / 1e6) * COST_INPUT + (total_out / 1e6) * COST_OUTPUT
    return {
        "gold": y_test, "pred": preds, "acc": acc,
        "in_tok": total_in, "out_tok": total_out, "cost": cost
    }

def run_hybrid_repetition(client, samples, ce_preds, ce_confs,
                          name, base_builder):
    """Experiment B/C: Hybrid with repetition on fallback samples only."""
    y_test = np.array([s["gold_label"] for s in samples])
    fallback_idx = np.where(ce_confs < FALLBACK_THRESHOLD)[0]
    hybrid_preds = ce_preds.copy()
    total_in, total_out = 0, 0

    print(f"  CE handled: {len(samples) - len(fallback_idx)}, "
          f"LLM fallback: {len(fallback_idx)}")

    for idx in tqdm(fallback_idx, desc=name):
        s = samples[idx]
        prompt = build_repetition(s["sentence1"], s["sentence2"], base_builder)
        raw, in_tok, out_tok = call_llm(client, prompt)
        hybrid_preds[idx] = parse_label(raw)
        total_in += in_tok
        total_out += out_tok
        time.sleep(REQUEST_DELAY)

    acc = accuracy_score(y_test, hybrid_preds)
    cost = (total_in / 1e6) * COST_INPUT + (total_out / 1e6) * COST_OUTPUT
    return {
        "gold": y_test.tolist(), "pred": hybrid_preds.tolist(), "acc": acc,
        "in_tok": total_in, "out_tok": total_out, "cost": cost,
        "api_calls": len(fallback_idx)
    }

# ============================================================
# Main
# ============================================================
def main():
    samples = load_data(DATA_FILE, TEST_SIZE)
    client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

    # Cross-Encoder (shared across hybrid experiments)
    ce_preds, ce_confs = run_cross_encoder(samples)

    all_results = {}

    # --- Experiment A: Pure LLM Zero-Shot Repetition ---
    print(f"\n{'='*60}")
    print("Experiment A: Pure LLM — Zero-Shot Repetition")
    print(f"{'='*60}")
    r = run_pure_llm_repetition(
        client, samples, "ZS-Rep (Pure)", build_zero_shot_base)
    all_results["Pure LLM ZS-Rep"] = r
    print(f"\nAccuracy: {r['acc']:.4f}")
    print(classification_report(
        r["gold"], r["pred"], labels=VALID_LABELS, zero_division=0))
    print_error_analysis(samples, r["gold"], r["pred"], "Pure LLM ZS-Rep")

    # --- Experiment B: Pure LLM Few-Shot Repetition ---
    print(f"\n{'='*60}")
    print("Experiment B: Pure LLM — Few-Shot Repetition")
    print(f"{'='*60}")
    r = run_pure_llm_repetition(
        client, samples, "FS-Rep (Pure)", build_few_shot_base)
    all_results["Pure LLM FS-Rep"] = r
    print(f"\nAccuracy: {r['acc']:.4f}")
    print(classification_report(
        r["gold"], r["pred"], labels=VALID_LABELS, zero_division=0))
    print_error_analysis(samples, r["gold"], r["pred"], "Pure LLM FS-Rep")

    # --- Experiment C: Hybrid + Zero-Shot Repetition ---
    print(f"\n{'='*60}")
    print("Experiment C: Hybrid — CE + Zero-Shot Repetition")
    print(f"{'='*60}")
    r = run_hybrid_repetition(
        client, samples, ce_preds, ce_confs,
        "Hybrid ZS-Rep", build_zero_shot_base)
    all_results["Hybrid ZS-Rep"] = r
    print(f"\nAccuracy: {r['acc']:.4f}")
    print(classification_report(
        r["gold"], r["pred"], labels=VALID_LABELS, zero_division=0))
    print_error_analysis(samples, r["gold"], r["pred"], "Hybrid ZS-Rep")

    # --- Experiment D: Hybrid + Few-Shot Repetition ---
    print(f"\n{'='*60}")
    print("Experiment D: Hybrid — CE + Few-Shot Repetition")
    print(f"{'='*60}")
    r = run_hybrid_repetition(
        client, samples, ce_preds, ce_confs,
        "Hybrid FS-Rep", build_few_shot_base)
    all_results["Hybrid FS-Rep"] = r
    print(f"\nAccuracy: {r['acc']:.4f}")
    print(classification_report(
        r["gold"], r["pred"], labels=VALID_LABELS, zero_division=0))
    print_error_analysis(samples, r["gold"], r["pred"], "Hybrid FS-Rep")

    # --- Summary ---
    print(f"\n{'='*60}")
    print("SUMMARY: Prompt Repetition Experiments")
    print(f"{'='*60}")
    rows = []
    for name, r in all_results.items():
        report = classification_report(
            r["gold"], r["pred"], labels=VALID_LABELS,
            output_dict=True, zero_division=0
        )
        rows.append({
            "Experiment": name,
            "Accuracy": r["acc"],
            "Macro-F1": report["macro avg"]["f1-score"],
            "Neutral-Recall": report["neutral"]["recall"],
            "API Calls": r.get("api_calls", len(samples)),
            "Input Tok": r["in_tok"],
            "Output Tok": r["out_tok"],
            "Cost ($)": r["cost"],
        })
    print(pd.DataFrame(rows).to_string(
        index=False, float_format="{:.4f}".format))

if __name__ == "__main__":
    main()